# T1 — API do modelo com FastAPI

**Objetivo:** transformar o `@champion` em um serviço. Qualquer aplicação manda uma hora do dia por HTTP
e recebe a previsão de aluguéis.

```
cliente ──► POST /predict {"hr": 18, "temp": 0.72, ...} ──► API (FastAPI) ──► @champion (MLflow no DagsHub)
        ◄── {"cnt": 842, "model_version": "1"} ◄──────────
```

| Conceito | O que é |
|---|---|
| **Endpoint** | um endereço + um verbo: `GET /health`, `POST /predict` |
| **Schema** | o contrato da entrada e da saída (classes do Pydantic) |
| **422** | a resposta da API quando a entrada não respeita o contrato |
| **`/docs`** | documentação interativa, gerada pelo FastAPI a partir dos schemas |

Toda resposta HTTP traz um **código de status**, além do corpo:

| Código | Significa | Quando aparece hoje |
|---|---|---|
| **200** | deu certo | `/health` e `/predict` com o `exemplo.json` |
| **422** | a entrada não respeita o contrato (o schema); o FastAPI recusa antes de chamar o modelo | falta um campo, ou um valor fora dos limites |
| **404** | o endereço não existe | um endereço digitado errado, como `/predicts` |
| **500** | erro dentro da API | o código da API quebrou |

> 💡 As células `%%writefile` gravam o arquivo inteiro: pode rodar de novo sem medo de duplicar nada.
> Abra o arquivo no editor para ver o resultado.

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(".env", override=True)   # endereço e credenciais do MLflow (T0)

for pasta in ["api", "tests"]:
    os.makedirs(pasta, exist_ok=True)

---
# Parte 1 · Juntos

## 1. A primeira versão da API

Três peças: o **schema** (`Hora`, as 12 entradas do modelo), o **carregamento** do `@champion` quando a API sobe
e os dois **endpoints**.

In [2]:
%%writefile api/main.py
"""API de previsão de demanda de bicicletas.

Rode da raiz do repositório:
    uvicorn api.main:app --port 8000 --reload
"""
from contextlib import asynccontextmanager

import mlflow
import pandas as pd
from dotenv import load_dotenv
from fastapi import FastAPI
from mlflow import MlflowClient
from pydantic import BaseModel, Field

# Endereço e credenciais do MLflow (DagsHub). No Codespace vêm do .env, que vale mais que o padrão
# do devcontainer. No Docker e no GitHub Actions não há .env: vêm das variáveis de ambiente.
load_dotenv(override=True)

MODEL_NAME = "bike-demand"
estado = {}  # o modelo carregado e a versão dele

class Hora(BaseModel):
    """As 12 entradas do modelo para uma hora do dia."""
    season: int
    yr: int
    mnth: int
    hr: int
    holiday: int
    weekday: int
    workingday: int
    weathersit: int
    temp: float
    atemp: float
    hum: float
    windspeed: float

class Previsao(BaseModel):
    cnt: int = Field(description="aluguéis previstos para a hora")
    model_version: str = Field(description="versão do modelo que respondeu")


def carregar_modelo():
    """O consumidor da T3 da Aula 1: só conhece o alias @champion.

    O endereço e as credenciais do MLflow vêm das variáveis de ambiente MLFLOW_TRACKING_*.
    """
    versao = MlflowClient().get_model_version_by_alias(MODEL_NAME, "champion").version
    modelo = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{versao}")
    return modelo, versao


@asynccontextmanager
async def lifespan(app: FastAPI):
    # Roda uma vez, quando a API sobe: carregar o modelo a cada requisição seria lento.
    estado["modelo"], estado["versao"] = carregar_modelo()
    yield


app = FastAPI(title="Bike Demand API", lifespan=lifespan)


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME, "model_version": estado["versao"]}


@app.post("/predict")
def predict(hora: Hora) -> Previsao:
    X = pd.DataFrame([hora.model_dump()]).astype("float64")
    cnt = float(estado["modelo"].predict(X)[0])
    # Regra de negócio: aluguel não é negativo nem fracionado.
    return Previsao(cnt=max(0, round(cnt)), model_version=estado["versao"])

Writing api/main.py


## 2. Subir a API

Em um **terminal** (deixe aberto, como o do MLflow na Aula 1):

```bash
uvicorn api.main:app --port 8000 --reload
```

- A API lê o `.env` sozinha (o `load_dotenv` no topo do `main.py`): é de lá que vêm o endereço e as
  credenciais do MLflow no DagsHub.
- `--reload` reinicia a API sozinho quando o arquivo muda.

🖱️ Na aba **PORTAS** (PORTS), clique no 🌐 da porta **8000** e acrescente `/docs` ao endereço.

## 3. Chamar a API

Primeiro o `/health`: a API está no ar? Com qual versão do modelo?

In [3]:
%%bash
curl -s -w "\nHTTP %{http_code}\n" localhost:8000/health

{"status":"ok","model":"bike-demand","model_version":"6"}
HTTP 200


Agora uma previsão. O exemplo é uma quarta-feira de agosto de 2012, às 18h, com tempo limpo:

In [4]:
%%writefile api/exemplo.json
{
  "season": 3, "yr": 1, "mnth": 8, "hr": 18, "holiday": 0, "weekday": 3,
  "workingday": 1, "weathersit": 1, "temp": 0.72, "atemp": 0.67, "hum": 0.45,
  "windspeed": 0.16
}

Writing api/exemplo.json


In [5]:
%%bash
curl -s -w "\nHTTP %{http_code}\n" -X POST localhost:8000/predict -H "Content-Type: application/json" -d @api/exemplo.json

{"cnt":852,"model_version":"6"}
HTTP 200


🖱️ Faça o mesmo pelo `/docs`: **POST /predict** → **Try it out** → cole o exemplo → **Execute**.

## 4. E se o cliente mandar uma entrada errada?

Dois casos: um campo faltando e uma hora que não existe (`hr = 25`).

In [6]:
%%bash
echo "--- sem o campo temp"
curl -s -w "\nHTTP %{http_code}\n" -X POST localhost:8000/predict -H "Content-Type: application/json" \
  -d '{"season": 3, "yr": 1, "mnth": 8, "hr": 18, "holiday": 0, "weekday": 3, "workingday": 1, "weathersit": 1, "atemp": 0.67, "hum": 0.45, "windspeed": 0.16}'
echo "--- hr = 25"
curl -s -w "\nHTTP %{http_code}\n" -X POST localhost:8000/predict -H "Content-Type: application/json" \
  -d '{"season": 3, "yr": 1, "mnth": 8, "hr": 25, "holiday": 0, "weekday": 3, "workingday": 1, "weathersit": 1, "temp": 0.72, "atemp": 0.67, "hum": 0.45, "windspeed": 0.16}'

--- sem o campo temp
{"detail":[{"type":"missing","loc":["body","temp"],"msg":"Field required","input":{"season":3,"yr":1,"mnth":8,"hr":18,"holiday":0,"weekday":3,"workingday":1,"weathersit":1,"atemp":0.67,"hum":0.45,"windspeed":0.16}}]}
HTTP 422
--- hr = 25
{"cnt":173,"model_version":"6"}
HTTP 200


> 💬 O campo faltando já é barrado (422): o schema exige os 12 campos. A hora 25 **passa** e o modelo responde
> um número com toda a confiança. Esse é o problema que a Parte 2 resolve.

## 5. O primeiro teste

O teste troca o modelo de verdade por um **modelo falso**, que responde sempre 123,4. Assim ele roda em
qualquer máquina, sem MLflow e sem rede. É o que a T3 vai precisar.

In [7]:
%%writefile pytest.ini
[pytest]
pythonpath = . src
testpaths = tests

Writing pytest.ini


In [9]:
%%writefile tests/test_api.py
"""Testes da API. Usam um modelo falso: não dependem do MLflow nem da rede."""
import json

import numpy as np
import pytest
from fastapi.testclient import TestClient

import api.main as main
from bike import FEATURES

EXEMPLO = json.load(open("api/exemplo.json"))


class ModeloFalso:
    """Responde sempre o mesmo valor: o que testamos aqui é a API, não o modelo."""

    def predict(self, X):
        return np.full(len(X), 123.4)


@pytest.fixture
def client(monkeypatch):
    monkeypatch.setattr(main, "carregar_modelo", lambda: (ModeloFalso(), "teste"))
    with TestClient(main.app) as c:
        yield c


def test_health(client):
    r = client.get("/health")
    assert r.status_code == 200
    assert r.json()["model_version"] == "teste"


def test_predict(client):
    r = client.post("/predict", json=EXEMPLO)
    assert r.status_code == 200
    assert r.json() == {"cnt": 123, "model_version": "teste"}

Overwriting tests/test_api.py


In [10]:
%%bash
pytest -q

..                                                                       [100%]
2 passed in 3.59s


### Como o teste funciona

O teste faz o mesmo que o `curl` do passo 4, mas em Python e sem precisar da API rodando no terminal:

| No terminal (passo 4) | No teste |
|---|---|
| `curl -X POST localhost:8000/predict -d @api/exemplo.json` | `r = client.post("/predict", json=EXEMPLO)` |
| você lê a resposta na tela | `r.status_code` (200, 422...) e `r.json()` (o corpo da resposta) |
| você confere se está certo | `assert r.status_code == 200`: se não for, o teste falha |

- `client` é a API rodando dentro do teste. A função `client` (uma *fixture* do `pytest`) troca o
  `carregar_modelo` pelo `ModeloFalso`, que responde sempre 123,4. Por isso o teste não precisa do DagsHub.
- `EXEMPLO` é o `api/exemplo.json`, lido como um dicionário do Python.

Na Parte 2 você vai escrever testes para as duas chamadas erradas do passo 4. Para isso, precisa montar a
entrada errada a partir do `EXEMPLO`. A célula abaixo mostra como:

In [11]:
import json

EXEMPLO = json.load(open("api/exemplo.json"))

hora_errada = {**EXEMPLO, "hr": 25}   # uma cópia do EXEMPLO, só com o hr trocado
print(EXEMPLO["hr"], "->", hora_errada["hr"])

sem_temp = EXEMPLO.copy()             # uma cópia do EXEMPLO...
del sem_temp["temp"]                  # ...sem o campo temp
print("temp" in EXEMPLO, "->", "temp" in sem_temp)

18 -> 25
True -> False


---
# Parte 2 · Sozinhos

## 1. Valide a entrada

O `Field` aceita limites: `ge` (maior ou igual) e `le` (menor ou igual). Complete os campos marcados com `TODO`.

| Campo | Valores válidos |
|---|---|
| `mnth` | 1 a 12 |
| `hr` | 0 a 23 |
| `holiday`, `workingday` | 0 ou 1 |
| `weekday` | 0 a 6 |
| `weathersit` | 1 a 4 |

In [23]:
%%writefile api/main.py
"""API de previsão de demanda de bicicletas.

Rode da raiz do repositório:
    uvicorn api.main:app --port 8000 --reload
"""
from contextlib import asynccontextmanager

import mlflow
import pandas as pd
from dotenv import load_dotenv
from fastapi import FastAPI
from mlflow import MlflowClient
from pydantic import BaseModel, Field

# Endereço e credenciais do MLflow (DagsHub). No Codespace vêm do .env, que vale mais que o padrão
# do devcontainer. No Docker e no GitHub Actions não há .env: vêm das variáveis de ambiente.
load_dotenv(override=True)

MODEL_NAME = "bike-demand"
estado = {}  # o modelo carregado e a versão dele

class Hora(BaseModel):
    """As 12 entradas do modelo para uma hora do dia."""
    season: int = Field(ge=1, le=4, description="estação do ano (1 a 4)")
    yr: int = Field(ge=0, le=1, description="ano (0 = 2011, 1 = 2012)")
    # TODO: troque os ... pelos limites (ge=..., le=...) de cada campo, como nos dois de cima
    mnth: int = Field(ge=1, le=12, description="mês")
    hr: int = Field(ge=0, le=23, description="hora do dia")
    holiday: int = Field(ge=0, le=1, description="feriado (0 ou 1)")
    weekday: int = Field(ge=0, le=6, description="dia da semana (0 = domingo)")
    workingday: int = Field(ge=0, le=1, description="dia útil (0 ou 1)")
    weathersit: int = Field(ge=1, le=4, description="clima (1 = limpo ... 4 = chuva forte)")
    temp: float = Field(ge=0, le=1, description="temperatura normalizada")
    atemp: float = Field(ge=0, le=1, description="sensação térmica normalizada")
    hum: float = Field(ge=0, le=1, description="umidade normalizada")
    windspeed: float = Field(ge=0, le=1, description="velocidade do vento normalizada")

class Previsao(BaseModel):
    cnt: int = Field(description="aluguéis previstos para a hora")
    model_version: str = Field(description="versão do modelo que respondeu")


def carregar_modelo():
    """O consumidor da T3 da Aula 1: só conhece o alias @champion.

    O endereço e as credenciais do MLflow vêm das variáveis de ambiente MLFLOW_TRACKING_*.
    """
    versao = MlflowClient().get_model_version_by_alias(MODEL_NAME, "champion").version
    modelo = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{versao}")
    return modelo, versao


@asynccontextmanager
async def lifespan(app: FastAPI):
    # Roda uma vez, quando a API sobe: carregar o modelo a cada requisição seria lento.
    estado["modelo"], estado["versao"] = carregar_modelo()
    yield


app = FastAPI(title="Bike Demand API", lifespan=lifespan)


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME, "model_version": estado["versao"]}


@app.post("/predict")
def predict(hora: Hora) -> Previsao:
    X = pd.DataFrame([hora.model_dump()]).astype("float64")
    cnt = float(estado["modelo"].predict(X)[0])
    # Regra de negócio: aluguel não é negativo nem fracionado.
    return Previsao(cnt=max(0, round(cnt)), model_version=estado["versao"])

Overwriting api/main.py


## 2. Repita a chamada com `hr = 25` (como no passo 4 da Parte 1)

In [24]:
%%bash
# TODO: repita a chamada com hr = 25. A resposta agora deve ser um erro 422.
echo "--- hr = 25"
curl -s -w "\nHTTP %{http_code}\n" -X POST localhost:8000/predict -H "Content-Type: application/json" \
  -d '{"season": 3, "yr": 1, "mnth": 8, "hr": 25, "holiday": 0, "weekday": 3, "workingday": 1, "weathersit": 1, "temp": 0.72, "atemp": 0.67, "hum": 0.45, "windspeed": 0.16}'

--- hr = 25
{"detail":[{"type":"less_than_equal","loc":["body","hr"],"msg":"Input should be less than or equal to 23","input":25,"ctx":{"le":23}}]}
HTTP 422


🖱️ Recarregue o `/docs` e abra o schema **Hora** no fim da página: os limites e as descrições aparecem lá.

## 3. Escreva os testes que faltam

Três testes novos. Os dois primeiros seguem o `test_predict` e usam as entradas erradas da célula
"Como o teste funciona" (passo 5 da Parte 1). O terceiro não chama a API: compara duas listas.

In [39]:
%%writefile tests/test_api.py
"""Testes da API. Usam um modelo falso: não dependem do MLflow nem da rede."""
import json

import numpy as np
import pytest
from fastapi.testclient import TestClient

import api.main as main
from bike import FEATURES

EXEMPLO = json.load(open("api/exemplo.json"))


class ModeloFalso:
    """Responde sempre o mesmo valor: o que testamos aqui é a API, não o modelo."""

    def predict(self, X):
        return np.full(len(X), 123.4)


@pytest.fixture
def client(monkeypatch):
    monkeypatch.setattr(main, "carregar_modelo", lambda: (ModeloFalso(), "teste"))
    with TestClient(main.app) as c:
        yield c


def test_health(client):
    r = client.get("/health")
    assert r.status_code == 200
    assert r.json()["model_version"] == "teste"


def test_predict(client):
    r = client.post("/predict", json=EXEMPLO)
    assert r.status_code == 200
    assert r.json() == {"cnt": 123, "model_version": "teste"}


def test_hora_invalida(client):
    hora_errada = {**EXEMPLO, "hr": 25}
    # TODO: mande a hora_errada (como o test_predict manda o EXEMPLO) e confira o código de erro
    r = client.post("/predict", json=hora_errada)
    assert r.status_code == 422


def test_campo_faltando(client):
    sem_temp = EXEMPLO.copy()
    del sem_temp["temp"]
    # TODO: mande o sem_temp e confira o código de erro
    r = client.post("/predict", json=sem_temp)
    assert r.status_code == 422


def test_contrato_com_o_treino():
    """A API e o treino precisam concordar sobre as entradas (nomes e ordem)."""
    # list(main.Hora.model_fields): os campos da classe Hora, na ordem em que foram escritos
    # FEATURES: as colunas que o treino usa (src/bike.py)
    assert list(main.Hora.model_fields) == FEATURES

Overwriting tests/test_api.py


## 4. Rode os testes e guarde (como no passo 5 da Parte 1)

In [42]:
%%bash
# TODO: rode os testes. Com os 5 passando, faça o commit de api/, tests/ e pytest.ini e o push.
pytest -q
git add api/tests/ pytest.ini
git commit -m "Adiciona API e testes"
git push



.....                                                                    [100%]
5 passed in 3.31s


fatal: pathspec 'api/tests/' did not match any files


On branch aula_02
Your branch is up to date with 'origin/aula_02'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   aula02/t1_api.ipynb
	modified:   tests/test_api.py

no changes added to commit (use "git add" and/or "git commit -a")


Everything up-to-date
